# Comparison SSL Phase 2: BIOT Fine-Tuning (WESAD)

## Overview
This notebook performs supervised fine-tuning of the **BIOT** model on the **WESAD** dataset. 
We use the pre-trained weights from Phase 1 to initialize the encoder, significantly improving the model's ability to detect stress from wrist-worn sensors (ACC, BVP, EDA, TEMP).

In [ ]:
# ── CELL 1: Setup and Architecture ──────────────────────────────────────────
import os, glob, pickle, torch, torch.nn as nn, numpy as np, pandas as pd
from scipy.signal import resample
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from einops import rearrange
from sklearn.metrics import accuracy_score, f1_score, classification_report

device = 'cuda' if torch.cuda.is_available() else 'cpu'
WESAD_DIR = glob.glob('/kaggle/input/**/WESAD', recursive=True)[0]
WEIGHTS_PATH = glob.glob('/kaggle/input/**/biot_encoder_pretrained.pt', recursive=True)
WEIGHTS_PATH = WEIGHTS_PATH[0] if WEIGHTS_PATH else '/kaggle/working/biot_encoder_pretrained.pt'

class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class BIOTEncoder(nn.Module):
    def __init__(self, n_channels=6, seg_len=640, chunk_size=64, d_model=256, n_heads=8, n_layers=4, dropout=0.1):
        super().__init__()
        self.n_chunks, self.chunk_size, self.d_model = seg_len // chunk_size, chunk_size, d_model
        self.patch_embed = nn.Linear(chunk_size, d_model); self.channel_emb = nn.Embedding(n_channels, d_model)
        self.pos_enc = SinusoidalPE(d_model, max_len=self.n_chunks * n_channels + 1); self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers); self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        B, C, T = x.shape
        x = rearrange(x[:, :, :self.n_chunks * self.chunk_size], 'b c (n p) -> b c n p', p=self.chunk_size)
        x = self.patch_embed(x) + self.channel_emb(torch.arange(C, device=x.device)).unsqueeze(0).unsqueeze(2)
        x = torch.cat([self.cls_token.expand(B, -1, -1), rearrange(x, 'b c n d -> b (c n) d')], dim=1)
        return self.norm(self.transformer(self.pos_enc(x)))[:, 0, :]

class BIOTClassifier(nn.Module):
    def __init__(self, encoder, n_classes=2, dropout=0.5):
        super().__init__()
        self.encoder = encoder; self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(encoder.d_model, encoder.d_model // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(encoder.d_model // 2, n_classes))
    def forward(self, x): return self.head(self.encoder(x))


### 2. Weight Loading and Data Preprocessing

In [ ]:
encoder = BIOTEncoder()
if os.path.exists(WEIGHTS_PATH):
    encoder.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
    print("Pre-trained weights loaded.")
model = BIOTClassifier(encoder).to(device)

TARGET_LEN, LABEL_MAP = 640, {1: 0, 2: 1}
TRAIN_SUBS = ['S2','S3','S4','S5','S6','S7','S8','S9','S10','S11']
TEST_SUBS  = ['S13','S14','S15','S16','S17']

def segment_subject(data, label_signal):
    wrist = data['signal']['wrist']
    bvp, eda, temp, acc = wrist['BVP'].flatten(), wrist['EDA'].flatten(), wrist['TEMP'].flatten(), wrist['ACC']
    lbl = resample(label_signal.astype(float), len(bvp)).round().astype(int)
    segs, labs = [], []
    for start in range(0, len(bvp) - TARGET_LEN + 1, TARGET_LEN):
        maj = int(np.bincount(lbl[start:start+TARGET_LEN][np.isin(lbl[start:start+TARGET_LEN], [1,2])]).argmax()) if len(lbl[start:start+TARGET_LEN][np.isin(lbl[start:start+TARGET_LEN], [1,2])]) > 0 else 0
        if maj not in LABEL_MAP: continue
        arr = np.stack([resample(acc[start//2:(start+TARGET_LEN)//2, i], TARGET_LEN) for i in range(3)] + [bvp[start:start+TARGET_LEN], resample(eda[start//16:(start+TARGET_LEN)//16], TARGET_LEN), resample(temp[start//16:(start+TARGET_LEN)//16], TARGET_LEN)])
        arr = (arr - arr.mean(axis=1, keepdims=True)) / (arr.std(axis=1, keepdims=True) + 1e-8)
        segs.append(arr); labs.append(LABEL_MAP[maj])
    return segs, labs

all_X, all_y, all_subs = [], [], []
for sdir in sorted(glob.glob(os.path.join(WESAD_DIR, 'S*'))):
    sid = os.path.basename(sdir)
    with open(os.path.join(sdir, f'{sid}.pkl'), 'rb') as f: data = pickle.load(f, encoding='latin1')
    segs, labs = segment_subject(data, data['label'])
    all_X.extend(segs); all_y.extend(labs); all_subs.extend([sid]*len(segs))

### 3. Fine-Tuning Loop

In [ ]:
u_subs = np.unique(all_subs); np.random.seed(42); np.random.shuffle(u_subs)
tr_s, te_s = u_subs[:int(0.7*len(u_subs))], u_subs[int(0.7*len(u_subs)):]
tr_idx = np.isin(all_subs, tr_s)

class SigDataset(Dataset):
    def __init__(self, m): self.X, self.y = torch.tensor(np.array(all_X)[m]), torch.tensor(np.array(all_y)[m])
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

weights = 1. / np.array([len(np.where(np.array(all_y)[tr_idx] == t)[0]) for t in np.unique(np.array(all_y)[tr_idx])])
sampler = WeightedRandomSampler(torch.tensor([weights[int(t)] for t in np.array(all_y)[tr_idx]]), len(np.where(tr_idx)[0]))
train_loader = DataLoader(SigDataset(tr_idx), batch_size=32, sampler=sampler)
test_loader = DataLoader(SigDataset(np.isin(all_subs, te_s)), batch_size=32)

opt = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.1); crit = nn.CrossEntropyLoss()
for e in range(1, 51):
    model.train()
    for xb, yb in train_loader: opt.zero_grad(); crit(model(xb.float().to(device)), yb.to(device)).backward(); opt.step()

model.eval(); p, t = [], []
with torch.no_grad():
    for xb, yb in test_loader: p.extend(model(xb.float().to(device)).argmax(1).cpu().numpy()); t.extend(yb.numpy())
print("Final Benchmarking:"); print(classification_report(t, p, target_names=['Baseline', 'Stress']))
acc = accuracy_score(t, p)
f1_stress = f1_score(t, p)
f1_macro = f1_score(t, p, average='macro')

import pandas as pd
df = pd.DataFrame({
    'Modello': ['Corponi (Master)', 'BIOT (Pre-trained)'],
    'Accuracy': ['80.20%', f'{acc*100:.2f}%'],
    'F1-Score (Stress)': ['54.55%', f'{f1_stress*100:.2f}%'],
    'Macro F1': ['70.95%', f'{f1_macro*100:.2f}%']
})
print('\n--- CONFRONTO FINALE WESAD ---')
print(df.to_string(index=False))

# --- Aggiunta Matrice di Confusione ---
if 'confusion_matrix' not in locals() and 'confusion_matrix' not in globals():
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    import matplotlib.pyplot as plt

cm = confusion_matrix(t, p)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Baseline', 'Stress'])
disp.plot(cmap='Blues')
plt.title('Matrice di Confusione')
plt.show()
